# Naive Bayes From Scratch With Numpy
Same spam task as spam_classifier.ipynb, but here I write multinomial naive bayes myself with numpy instead of using sklearn. At the end I check if I get the same result as sklearn's MultinomialNB.

In [ ]:
import numpy as np
from sklearn.naive_bayes import MultinomialNB # only to compare at the end
from sklearn.metrics import accuracy_score, confusion_matrix

from data_prep import read_mails, build_dictionary, to_vectors # same functions as spam_classifier.ipynb

## Prepare Data
Same steps as in spam_classifier.ipynb, the functions are in data_prep.py. Read the mails, make the dictionary from train and turn every mail into a word count vector.

In [ ]:
train_texts, train_labels = read_mails('train-mails')
test_texts, test_labels = read_mails('test-mails')

len(train_texts), sum(train_labels), len(test_texts), sum(test_labels) # should be 702, 351, 260, 130

In [ ]:
top_words = build_dictionary(train_texts) # only from train
word_to_index = {word: index for index, word in enumerate(top_words)}

In [ ]:
X_train = to_vectors(train_texts, word_to_index)
X_test = to_vectors(test_texts, word_to_index)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

X_train.shape, X_test.shape

## How Multinomial Naive Bayes Works

For a mail with word counts $x_1, \dots, x_n$ we want the class $c$ (ham or spam) with the highest

$$P(c \mid x) \propto P(c) \prod_{w} P(w \mid c)^{x_w}$$

Prior, how common the class is in train:

$$P(c) = \frac{\text{number of mails in } c}{\text{number of all mails}}$$

Likelihood of a word in a class, with laplace smoothing $\alpha$ so a word that never shows up in a class doesnt make everything 0:

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha \cdot V}$$

$V$ is the dictionary size (2000).

Multiplying 2000 small probabilities underflows to 0, so we use logs and the product becomes a sum:

$$\log P(c) + \sum_w x_w \log P(w \mid c)$$

For all mails at once this is just a matrix multiplication: `X @ log_likelihoods.T + log_priors`. Then the class with the bigger score wins (argmax).

## Implement It

In [ ]:
class MyMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha # smoothing

    def fit(self, X, y):
        self.classes = np.unique(y) # [0, 1]
        n_classes = len(self.classes)
        n_words = X.shape[1]

        self.class_log_prior = np.zeros(n_classes)
        self.feature_log_prob = np.zeros((n_classes, n_words))

        for i, c in enumerate(self.classes):
            X_c = X[y == c] # only the mails of this class

            self.class_log_prior[i] = np.log(len(X_c) / len(X))

            word_counts = X_c.sum(axis=0) + self.alpha # how often each word shows up in this class, plus smoothing
            self.feature_log_prob[i] = np.log(word_counts / word_counts.sum())

        return self

    def scores(self, X):
        # log P(c) + sum of x_w * log P(w|c), one column per class
        return X @ self.feature_log_prob.T + self.class_log_prior

    def predict(self, X):
        return self.classes[np.argmax(self.scores(X), axis=1)]

## Train And Test

In [ ]:
my_model = MyMultinomialNB(alpha=1.0)
my_model.fit(X_train, y_train)

my_pred = my_model.predict(X_test)

accuracy_score(y_test, my_pred)

In [ ]:
# rows are the real class (ham, spam), columns are predicted (ham, spam)
confusion_matrix(y_test, my_pred)

## Compare With sklearn
If my implementation is right, the priors, the word log probabilities and the predictions should all be the same as sklearn's.

In [ ]:
sk_model = MultinomialNB(alpha=1.0)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_test)

print('same priors:', np.allclose(my_model.class_log_prior, sk_model.class_log_prior_))
print('same word probabilities:', np.allclose(my_model.feature_log_prob, sk_model.feature_log_prob_))
print('same predictions:', (my_pred == sk_pred).all())
print('sklearn accuracy:', accuracy_score(y_test, sk_pred))

## One Mail Step By Step
Looking at the scores of one test mail to see what the model actually compares.

In [ ]:
i = 0 # first test mail
x = X_test[i]

ham_score = my_model.class_log_prior[0] + x @ my_model.feature_log_prob[0]
spam_score = my_model.class_log_prior[1] + x @ my_model.feature_log_prob[1]

print('real label:', y_test[i])
print('ham score:', ham_score)
print('spam score:', spam_score)
print('prediction:', int(spam_score > ham_score))

## Most Spammy Words
log P(w|spam) - log P(w|ham). Big positive means the word pushes a mail to spam, big negative pushes it to ham.

In [ ]:
spamminess = my_model.feature_log_prob[1] - my_model.feature_log_prob[0]
order = np.argsort(spamminess)

print('most spammy:', [top_words[j] for j in order[::-1][:15]])
print('most hammy:', [top_words[j] for j in order[:15]])

## Results
My implementation gives exactly the same priors, word probabilities and predictions as sklearn, so the accuracy is the same too, 96.5%.

The whole model is just counting words per class, adding alpha, taking logs and one matrix multiplication.